# 🔍 Optimización — S1

## Federated Proactive Forest

**Estrategia:** Pools ALL trees from ALL clients. No selection.

**Hiperparámetros:** `alpha_pf`, `local_weight`

**Datasets:** Letter, Optdigits, Spambase, Nursery, Sonar, Vowel

**N_CLIENTS:** 3

> ⚡ **Cada celda de dataset es independiente** — ejecuta solo la que necesites.

> 📋 **Rangos leídos en runtime desde** `configs/experiments/optimization/search_spaces.yaml`

In [2]:
# ── Imports & Config ─────────────────────────────────────────────────────
import sys
from pathlib import Path

# Suppress tqdm progress bar warning in Jupyter (ipywidgets not installed)
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm')

ROOT = Path.cwd().parent.parent.parent.parent
sys.path.insert(0, str(ROOT))

from src.infrastructure.dataset.dataset_factory import DatasetFactory

import numpy as np
import pandas as pd
import yaml
import json
import optuna
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split

SEED = 42
N_CLIENTS = 3
N_TRIALS = 20
STRATEGY = 'S1'
DATA_DIR = ROOT / 'data'
RESULTS_DIR = ROOT / 'results' / 's6_alpha_optimization'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load search space from YAML (single source of truth)
with open(ROOT / 'configs' / 'experiments' / 'optimization' / 'search_spaces.yaml') as f:
    all_spaces = yaml.safe_load(f)
SPACE = all_spaces[STRATEGY]

from src.domain.dataset.base_adapter import DatasetSplit
from src.application.orchestrators.fl_orchestrator import FLEXOrchestrator

print(f'✅ Project root: {ROOT}')
print(f'✅ Strategy: {STRATEGY}')
print(f'✅ N_CLIENTS: {N_CLIENTS}')
print(f'✅ Search space: {list(SPACE.keys())}')

c:\Users\Adrián Rodríguez\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Project root: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest
✅ Strategy: S1
✅ N_CLIENTS: 3
✅ Search space: ['alpha_pf', 'local_weight']


## Dataset: Optdigits

5,620 samples, 64 features (8x8 pixel), 10 classes (0-9), numeric 0-16

In [2]:
# ── Load Optdigits ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'optdigits.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='optdigits')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S1 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S1 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'optdigits',
    'strategy': 'S1',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_optdigits_s1_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_optdigits_s1_results.json")

📊 Shape: (5620, 65)
✅ Train=4496, Test=1124, Feats=64, Classes=10
🚀 Optimizando S1 en optdigits... (20 trials)


[I 2026-04-05 14:09:20,777] A new study created in memory with name: no-name-3a1148cb-6737-43f7-a088-57298f3901de



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 63 árboles entrenados
  client_2: 70 árboles entrenados
  TOTAL: 149 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 63/63 árboles seleccionados
  client_2: 70/70 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 149 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 133 globales externos (16 propios excluidos) = 149 árboles
  client_1: 63 locales + 86 globales externos (63 propios excluidos) = 149 árboles
  client_2: 70 locales + 79 globales externos (70 propios excluidos) = 149 árboles



[I 2026-04-05 14:13:18,293] Trial 0 finished with value: 0.9670079591512017 and parameters: {'alpha_pf': 0.35, 'local_weight': 1.0}. Best is trial 0 with value: 0.9670079591512017.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 37 árboles entrenados
  client_2: 48 árboles entrenados
  TOTAL: 101 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 37/37 árboles seleccionados
  client_2: 48/48 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 101 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 85 globales externos (16 propios excluidos) = 101 árboles
  client_1: 37 locales + 64 globales externos (37 propios excluidos) = 101 árboles
  client_2: 48 locales + 53 globales externos (48 propios excluidos) = 101 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 33 árboles entrenados
  client_1: 75 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 124 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 33/33 árboles seleccionados
  client_1: 75/75 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 124 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 33 locales + 91 globales externos (33 propios excluidos) = 124 árboles
  client_1: 75 locales + 49 globales externos (75 propios excluidos) = 124 árboles
  client_2: 16 locales + 108 globales externos (16 propios excluidos) = 124 árboles



[I 2026-04-05 15:15:24,044] Trial 18 finished with value: 0.9661936576544952 and parameters: {'alpha_pf': 0.65, 'local_weight': 0.8}. Best is trial 15 with value: 0.9724881422303409.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 100 árboles entrenados
  client_1: 61 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 177 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 100/100 árboles seleccionados
  client_1: 61/61 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 177 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 100 locales + 77 globales externos (100 propios excluidos) = 177 árboles
  client_1: 61 locales + 116 globales externos (61 propios excluidos) = 177 árboles
  client_2: 16 locales + 161 globales externos (16 propios excluidos) = 177 árboles



[I 2026-04-05 15:23:15,697] Trial 19 finished with value: 0.9680213406775726 and parameters: {'alpha_pf': 0.4, 'local_weight': 0.5}. Best is trial 15 with value: 0.9724881422303409.



📊 S1 — optdigits — Resultados
   Mejor Macro-F1: 0.9725
   alpha_pf                 : 0.45000000000000007
   local_weight             : 0.6000000000000001
   Media Macro-F1:  0.9681
   Std Macro-F1:    0.0027

✅ Resultados guardados: s6_optdigits_s1_results.json


## Dataset: Spambase

4,601 samples, 57 features (word frequencies), 2 classes (spam/ham)

In [3]:
# ── Load Spambase ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'spambase.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='spambase')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S1 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S1 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'spambase',
    'strategy': 'S1',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_spambase_s1_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_spambase_s1_results.json")

📊 Shape: (4601, 58)
✅ Train=3680, Test=921, Feats=57, Classes=2
🚀 Optimizando S1 en spambase... (20 trials)


[I 2026-04-05 20:19:20,701] A new study created in memory with name: no-name-0e957e33-af23-4707-8fe7-72439827b7ca



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 46 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 86 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 46/46 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 86 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 70 globales externos (16 propios excluidos) = 86 árboles
  client_1: 46 locales + 40 globales externos (46 propios excluidos) = 86 árboles
  client_2: 24 locales + 62 globales externos (24 propios excluidos) = 86 árboles



[I 2026-04-05 20:20:38,674] Trial 0 finished with value: 0.9185967332132449 and parameters: {'alpha_pf': 0.35, 'local_weight': 1.0}. Best is trial 0 with value: 0.9185967332132449.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 33 árboles entrenados
  client_2: 27 árboles entrenados
  TOTAL: 84 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 24/24 árboles seleccionados
  client_1: 33/33 árboles seleccionados
  client_2: 27/27 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 84 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 60 globales externos (24 propios excluidos) = 84 árboles
  client_1: 33 locales + 51 globales externos (33 propios excluidos) = 84 árboles
  client_2: 27 locales + 57 globales externos (27 propios excluidos) = 84 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 24 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 72 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 24/24 árboles seleccionados
  client_1: 24/24 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 72 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 48 globales externos (24 propios excluidos) = 72 árboles
  client_1: 24 locales + 48 globales externos (24 propios excluidos) = 72 árboles
  client_2: 24 locales + 48 globales externos (24 propios excluidos) = 72 árboles



[I 2026-04-05 21:24:51,789] Trial 18 finished with value: 0.9201945929887106 and parameters: {'alpha_pf': 0.2, 'local_weight': 0.4}. Best is trial 2 with value: 0.930207943829451.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 63 árboles entrenados
  client_1: 27 árboles entrenados
  client_2: 88 árboles entrenados
  TOTAL: 178 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 63/63 árboles seleccionados
  client_1: 27/27 árboles seleccionados
  client_2: 88/88 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 178 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 63 locales + 115 globales externos (63 propios excluidos) = 178 árboles
  client_1: 27 locales + 151 globales externos (27 propios excluidos) = 178 árboles
  client_2: 88 locales + 90 globales externos (88 propios excluidos) = 178 árboles



[I 2026-04-05 21:27:56,556] Trial 19 finished with value: 0.9234997675499768 and parameters: {'alpha_pf': 0.75, 'local_weight': 0.1}. Best is trial 2 with value: 0.930207943829451.



📊 S1 — spambase — Resultados
   Mejor Macro-F1: 0.9302
   alpha_pf                 : 0.2
   local_weight             : 0.1
   Media Macro-F1:  0.9239
   Std Macro-F1:    0.0034

✅ Resultados guardados: s6_spambase_s1_results.json


## Dataset: Nursery

12,960 samples, 8 categorical features, 5 classes

In [4]:
# ── Load Nursery ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'nursery.csv')
print(f'📊 Shape: {df.shape}')
cat_cols = ['parents', 'has_nurs', 'form', 'children', 'housing', 'finance', 'social', 'health']
enc = OrdinalEncoder()
X = enc.fit_transform(df[cat_cols])
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='nursery')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S1 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S1 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'nursery',
    'strategy': 'S1',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_nursery_s1_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_nursery_s1_results.json")

📊 Shape: (12960, 9)
✅ Train=10368, Test=2592, Feats=8, Classes=5
🚀 Optimizando S1 en nursery... (20 trials)


[I 2026-04-05 21:31:52,695] A new study created in memory with name: no-name-d053e4a5-9738-4645-83d3-41417e8d22b4



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 38 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 86 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 24/24 árboles seleccionados
  client_1: 38/38 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 86 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 62 globales externos (24 propios excluidos) = 86 árboles
  client_1: 38 locales + 48 globales externos (38 propios excluidos) = 86 árboles
  client_2: 24 locales + 62 globales externos (24 propios excluidos) = 86 árboles



[I 2026-04-05 21:32:36,911] Trial 0 finished with value: 0.9566734242485421 and parameters: {'alpha_pf': 0.35, 'local_weight': 1.0}. Best is trial 0 with value: 0.9566734242485421.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 33 árboles entrenados
  client_1: 27 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 76 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 33/33 árboles seleccionados
  client_1: 27/27 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 76 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 33 locales + 43 globales externos (33 propios excluidos) = 76 árboles
  client_1: 27 locales + 49 globales externos (27 propios excluidos) = 76 árboles
  client_2: 16 locales + 60 globales externos (16 propios excluidos) = 76 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 72 árboles entrenados
  client_2: 50 árboles entrenados
  TOTAL: 138 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 72/72 árboles seleccionados
  client_2: 50/50 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 138 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 122 globales externos (16 propios excluidos) = 138 árboles
  client_1: 72 locales + 66 globales externos (72 propios excluidos) = 138 árboles
  client_2: 50 locales + 88 globales externos (50 propios excluidos) = 138 árboles



[I 2026-04-05 21:50:38,608] Trial 18 finished with value: 0.9547794662295298 and parameters: {'alpha_pf': 0.5, 'local_weight': 0.8}. Best is trial 1 with value: 0.9603337623283645.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 69 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 117 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 24/24 árboles seleccionados
  client_1: 69/69 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 117 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 93 globales externos (24 propios excluidos) = 117 árboles
  client_1: 69 locales + 48 globales externos (69 propios excluidos) = 117 árboles
  client_2: 24 locales + 93 globales externos (24 propios excluidos) = 117 árboles



[I 2026-04-05 21:51:48,962] Trial 19 finished with value: 0.9591344935372138 and parameters: {'alpha_pf': 0.75, 'local_weight': 0.5}. Best is trial 1 with value: 0.9603337623283645.



📊 S1 — nursery — Resultados
   Mejor Macro-F1: 0.9603
   alpha_pf                 : 0.6
   local_weight             : 0.6000000000000001
   Media Macro-F1:  0.9524
   Std Macro-F1:    0.0069

✅ Resultados guardados: s6_nursery_s1_results.json


## Dataset: Sonar

208 samples, 60 numeric features, 2 classes (Rock/Mine) — small dataset!

In [5]:
# ── Load Sonar ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'sonar.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['Class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['Class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'Class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='sonar')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S1 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S1 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'sonar',
    'strategy': 'S1',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_sonar_s1_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_sonar_s1_results.json")

📊 Shape: (208, 61)
✅ Train=166, Test=42, Feats=60, Classes=2
🚀 Optimizando S1 en sonar... (20 trials)


[I 2026-04-05 21:55:13,286] A new study created in memory with name: no-name-c39c0371-b06c-4f08-8f5f-9b27b527f31b



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 33 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 65 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 33/33 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 65 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 49 globales externos (16 propios excluidos) = 65 árboles
  client_1: 33 locales + 32 globales externos (33 propios excluidos) = 65 árboles
  client_2: 16 locales + 49 globales externos (16 propios excluidos) = 65 árboles



[I 2026-04-05 21:55:21,422] Trial 0 finished with value: 0.71880706025563 and parameters: {'alpha_pf': 0.35, 'local_weight': 1.0}. Best is trial 0 with value: 0.71880706025563.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 38 árboles entrenados
  client_1: 37 árboles entrenados
  client_2: 33 árboles entrenados
  TOTAL: 108 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 38/38 árboles seleccionados
  client_1: 37/37 árboles seleccionados
  client_2: 33/33 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 108 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 38 locales + 70 globales externos (38 propios excluidos) = 108 árboles
  client_1: 37 locales + 71 globales externos (37 propios excluidos) = 108 árboles
  client_2: 33 locales + 75 globales externos (33 propios excluidos) = 108 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 37 árboles entrenados
  client_1: 24 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 77 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 37/37 árboles seleccionados
  client_1: 24/24 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 77 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 37 locales + 40 globales externos (37 propios excluidos) = 77 árboles
  client_1: 24 locales + 53 globales externos (24 propios excluidos) = 77 árboles
  client_2: 16 locales + 61 globales externos (16 propios excluidos) = 77 árboles



[I 2026-04-05 21:58:27,962] Trial 18 finished with value: 0.8253119429590018 and parameters: {'alpha_pf': 0.65, 'local_weight': 0.8}. Best is trial 1 with value: 0.851764705882353.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 27 árboles entrenados
  TOTAL: 59 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 27/27 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 59 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 43 globales externos (16 propios excluidos) = 59 árboles
  client_1: 16 locales + 43 globales externos (16 propios excluidos) = 59 árboles
  client_2: 27 locales + 32 globales externos (27 propios excluidos) = 59 árboles



[I 2026-04-05 21:58:33,870] Trial 19 finished with value: 0.8309373202990225 and parameters: {'alpha_pf': 0.45000000000000007, 'local_weight': 0.4}. Best is trial 1 with value: 0.851764705882353.



📊 S1 — sonar — Resultados
   Mejor Macro-F1: 0.8518
   alpha_pf                 : 0.6
   local_weight             : 0.6000000000000001
   Media Macro-F1:  0.7939
   Std Macro-F1:    0.0375

✅ Resultados guardados: s6_sonar_s1_results.json


## Dataset: Vowel

990 samples, 10 numeric features (drop 3 metadata cols), 11 classes

In [6]:
# ── Load Vowel ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'vowel.csv')
print(f'📊 Shape: {df.shape}')
df = df.drop(columns=['Train or Test', 'Speaker Number', 'Sex'])
X = df.drop(columns=['Class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['Class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'Class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='vowel')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S1 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S1 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'vowel',
    'strategy': 'S1',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_vowel_s1_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_vowel_s1_results.json")

📊 Shape: (990, 14)
✅ Train=792, Test=198, Feats=10, Classes=11
🚀 Optimizando S1 en vowel... (20 trials)


[I 2026-04-05 22:16:07,377] A new study created in memory with name: no-name-88ef4f78-fa19-43a4-ae2b-c1ba53d2f19d



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 35 árboles entrenados
  client_2: 100 árboles entrenados
  TOTAL: 159 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 24/24 árboles seleccionados
  client_1: 35/35 árboles seleccionados
  client_2: 100/100 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 159 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 135 globales externos (24 propios excluidos) = 159 árboles
  client_1: 35 locales + 124 globales externos (35 propios excluidos) = 159 árboles
  client_2: 100 locales + 59 globales externos (100 propios excluidos) = 159 árboles



[I 2026-04-05 22:16:49,422] Trial 0 finished with value: 0.7762304579759859 and parameters: {'alpha_pf': 0.35, 'local_weight': 1.0}. Best is trial 0 with value: 0.7762304579759859.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 24 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 64 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 24/24 árboles seleccionados
  client_1: 24/24 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 64 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 40 globales externos (24 propios excluidos) = 64 árboles
  client_1: 24 locales + 40 globales externos (24 propios excluidos) = 64 árboles
  client_2: 16 locales + 48 globales externos (16 propios excluidos) = 64 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 100 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 132 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 100/100 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 132 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 116 globales externos (16 propios excluidos) = 132 árboles
  client_1: 100 locales + 32 globales externos (100 propios excluidos) = 132 árboles
  client_2: 16 locales + 116 globales externos (16 propios excluidos) = 132 árboles



[I 2026-04-05 22:25:46,425] Trial 18 finished with value: 0.7591459284535559 and parameters: {'alpha_pf': 0.45000000000000007, 'local_weight': 0.6000000000000001}. Best is trial 5 with value: 0.84369901188083.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 38 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 78 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 38/38 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 78 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 62 globales externos (16 propios excluidos) = 78 árboles
  client_1: 38 locales + 40 globales externos (38 propios excluidos) = 78 árboles
  client_2: 24 locales + 54 globales externos (24 propios excluidos) = 78 árboles



[I 2026-04-05 22:26:11,960] Trial 19 finished with value: 0.8020513938189141 and parameters: {'alpha_pf': 0.7000000000000001, 'local_weight': 0.9}. Best is trial 5 with value: 0.84369901188083.



📊 S1 — vowel — Resultados
   Mejor Macro-F1: 0.8437
   alpha_pf                 : 0.1
   local_weight             : 1.0
   Media Macro-F1:  0.7945
   Std Macro-F1:    0.0193

✅ Resultados guardados: s6_vowel_s1_results.json


## Dataset: Letter

20,000 samples, 16 features, 26 classes (A-Z), numeric features — LARGEST, runs last

In [7]:
# ── Load Letter ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'letter.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='letter')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S1 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S1 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'letter',
    'strategy': 'S1',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_letter_s1_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_letter_s1_results.json")

📊 Shape: (20000, 17)
✅ Train=16000, Test=4000, Feats=16, Classes=26
🚀 Optimizando S1 en letter... (20 trials)


[I 2026-04-05 22:26:12,326] A new study created in memory with name: no-name-a2595928-056a-4d52-bc37-d0694f24edc8



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 76 árboles entrenados
  client_2: 27 árboles entrenados
  TOTAL: 127 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 24/24 árboles seleccionados
  client_1: 76/76 árboles seleccionados
  client_2: 27/27 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 127 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 103 globales externos (24 propios excluidos) = 127 árboles
  client_1: 76 locales + 51 globales externos (76 propios excluidos) = 127 árboles
  client_2: 27 locales + 100 globales externos (27 propios excluidos) = 127 árboles



[I 2026-04-05 22:31:54,741] Trial 0 finished with value: 0.9214797438066977 and parameters: {'alpha_pf': 0.35, 'local_weight': 1.0}. Best is trial 0 with value: 0.9214797438066977.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 27 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 59 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 27/27 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 59 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 43 globales externos (16 propios excluidos) = 59 árboles
  client_1: 27 locales + 32 globales externos (27 propios excluidos) = 59 árboles
  client_2: 16 locales + 43 globales externos (16 propios excluidos) = 59 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 37 árboles entrenados
  client_2: 38 árboles entrenados
  TOTAL: 91 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 37/37 árboles seleccionados
  client_2: 38/38 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 91 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 75 globales externos (16 propios excluidos) = 91 árboles
  client_1: 37 locales + 54 globales externos (37 propios excluidos) = 91 árboles
  client_2: 38 locales + 53 globales externos (38 propios excluidos) = 91 árboles



[I 2026-04-05 23:58:01,964] Trial 18 finished with value: 0.9189426980766551 and parameters: {'alpha_pf': 0.45000000000000007, 'local_weight': 0.2}. Best is trial 6 with value: 0.9289337935659708.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 27 árboles entrenados
  client_2: 38 árboles entrenados
  TOTAL: 81 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 27/27 árboles seleccionados
  client_2: 38/38 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 81 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 65 globales externos (16 propios excluidos) = 81 árboles
  client_1: 27 locales + 54 globales externos (27 propios excluidos) = 81 árboles
  client_2: 38 locales + 43 globales externos (38 propios excluidos) = 81 árboles



[I 2026-04-06 00:00:50,954] Trial 19 finished with value: 0.9215040313480062 and parameters: {'alpha_pf': 0.7000000000000001, 'local_weight': 0.4}. Best is trial 6 with value: 0.9289337935659708.



📊 S1 — letter — Resultados
   Mejor Macro-F1: 0.9289
   alpha_pf                 : 0.7000000000000001
   local_weight             : 0.2
   Media Macro-F1:  0.9214
   Std Macro-F1:    0.0033

✅ Resultados guardados: s6_letter_s1_results.json


## 📊 Resumen Global — Comparar todos los datasets

> ⚡ Ejecuta esta celda **después** de haber ejecutado todas las celdas de datasets.

In [9]:
# ── Resumen Global (ejecutar después de todas las celdas) ─────────────
results = []
for fp in sorted(RESULTS_DIR.glob('s6_*_s1_results.json')):
    with open(fp) as f:
        r = json.load(f)
    row = {'dataset': r['dataset'], 'best_f1': round(r['best_macro_f1'], 4),
           'mean_f1': round(r['mean_macro_f1'], 4), 'std': round(r['std_macro_f1'], 4)}
    row.update(r['best_params'])
    results.append(row)

df_sum = pd.DataFrame(results)
print(f"\n{'='*90}")
print(f"📋 RESUMEN GLOBAL — S1")
print(f"{'='*90}")
if df_sum.empty:
    print('⚠️ No hay resultados. Ejecuta al menos una celda de dataset primero.')
else:
    print(df_sum.to_string(index=False))
    if 'alpha_pf' in df_sum.columns:
        rec = df_sum['alpha_pf'].median()
        print(f"\n🎯 alpha_pf recomendado (mediana): {rec:.1f}")
        print(f"   Rango: [{df_sum['alpha_pf'].min()} — {df_sum['alpha_pf'].max()}]")
    df_sum.to_csv(RESULTS_DIR / 'summary_s1.csv', index=False)
    print(f"\n✅ Resumen guardado: summary_s1.csv")


📋 RESUMEN GLOBAL — S1
  dataset  best_f1  mean_f1    std  alpha_pf  local_weight
   letter   0.9289   0.9214 0.0033      0.70           0.2
  nursery   0.9603   0.9524 0.0069      0.60           0.6
optdigits   0.9725   0.9681 0.0027      0.45           0.6
    sonar   0.8518   0.7939 0.0375      0.60           0.6
 spambase   0.9302   0.9239 0.0034      0.20           0.1
    vowel   0.8437   0.7945 0.0193      0.10           1.0

🎯 alpha_pf recomendado (mediana): 0.5
   Rango: [0.1 — 0.7000000000000001]

✅ Resumen guardado: summary_s1.csv
